# NAS-Bench-201 — revised pipeline (leakage-free)

Reruns Case Study 1 and Case Study 2 under the protocol the reviewers asked for.
Replaces `NAS-pipeline.ipynb`, `NAS-mo-pipeline.ipynb` and
`NAS-mo-pipeline-gde3-qehvi.ipynb`, which all searched on **test** accuracy.

**What changes**

| | old | new |
|---|---|---|
| Search signal | `test_acc` | `valid_acc` (`cifar10-valid` for CIFAR-10) |
| Test accuracy | optimized against | read once, for the returned architecture |
| Duplicate lookups | re-evaluated, uncached, per-method | one shared cache; proposals **and** unique both reported |
| Scalarization | weighted sum, α ∈ [0.1, 0.9] | weighted sum **and** Tchebycheff, α ∈ [0, 1] |
| Baselines | — | random search, regularized evolution |

**How to use it.** Run top to bottom. Stages are gated by flags in the config
cell and default to the cheap ones. Every run is checkpointed to
`manifest.csv`, so re-running the notebook skips finished work — safe to
interrupt and resume.

**Do stage 2 first and stop.** It costs minutes. Do not start stage 3 (tens of
hours) until you have looked at what stage 2 produced.

## Environment

In [ ]:
import json, os, sys, time, warnings
from pathlib import Path
from datetime import datetime

import numpy as np
import pandas as pd

warnings.filterwarnings('ignore')
pd.set_option('display.width', 200)

ROOT = Path.cwd()
if not (ROOT / 'utils').exists() and (ROOT.parent / 'utils').exists():
    ROOT = ROOT.parent
sys.path.insert(0, str(ROOT))

from utils.nas import getApi
from utils.nas_revision import (
    ALPHA_SWEEP_DENSE, REF_POINT, N_OPS, N_EDGES,
    trueParetoFront, hypervolume, igdPlus, scoreRun, splitFor,
    runNSGA2MO, runGDE3MO, runQEHVIMO,
    runScalarizedMealpy, runBOScalarized,
    runRandomSearch, runRegularizedEvolution,
    writeLegacyTrueFront,
)
from utils.nas_figures import makeAll as makeFigures

# Folder names that aggregate_mo_nas.py already expects, so the existing
# figure/summary script reads the new results with no changes.
ALGO_FOLDERS = {
    'GA': 'ga', 'SADE': 'sade', 'AIW_PSO': 'aiw_pso', 'GWO': 'gwo',
    'BO': 'bo', 'NSGA-II': 'nsga2', 'GDE3': 'gde3', 'qEHVI': 'qehvi',
    'RandomSearch': 'random_search',
    'RegularizedEvolution': 'regularized_evolution',
}

# Which optional backends are present decides which stages can run.
HAVE = {}
for mod in ('nats_bench', 'mealpy', 'bayes_opt', 'botorch', 'pymoo', 'scipy'):
    try:
        __import__(mod)
        HAVE[mod] = True
    except ImportError:
        HAVE[mod] = False

print('ROOT :', ROOT)
for k, v in HAVE.items():
    print(f'  {k:12} {"ok" if v else "MISSING"}')

missing = [k for k, v in HAVE.items() if not v]
if missing:
    print('\ninstall with:  pip install ' + ' '.join(
        {'bayes_opt': 'bayesian-optimization'}.get(m, m) for m in missing))

In [ ]:
NAS_PATH = ROOT / 'utils' / 'nas_records' / 'NATS-tss-v1_0' / 'NATS-tss-v1_0'
assert NAS_PATH.exists(), f'benchmark not found at {NAS_PATH}'

api = getApi(NAS_PATH)
print('benchmark loaded')
for ds in ('cifar10', 'cifar100', 'ImageNet16-120'):
    s_ds, s_key, r_ds, r_key = splitFor(ds)
    print(f'  {ds:16} search on {s_ds:16} ({s_key})   report on {r_ds:16} ({r_key})')

## Configuration

In [ ]:
DATASETS  = ['cifar10', 'cifar100', 'ImageNet16-120']
HP        = '200'
BUDGET    = 1020
POP_SIZE  = 20
SEEDS_MO  = [0, 1, 2]           # multi-objective (matches the paper)
SEEDS_SO  = [0, 1, 2, 3, 4]     # single-objective (matches the paper)

# The budget is BUDGET objective calls, repeats included — the standard
# optimization-benchmarking convention. The optimizer decides what to explore
# and we never intervene when it re-proposes an architecture it has already
# seen. The lookup is memoized, so a repeat costs no time; it simply yields no
# new information. How much of its budget each method spends on distinct
# architectures is therefore a RESULT about its search behaviour, reported in
# the proposals-vs-unique figure, not something to correct away.

SCALARIZATIONS = ['weighted_sum', 'tchebycheff']
EAS = ['GA', 'SADE', 'GWO', 'AIW_PSO']   # NB: AIW_PSO, not PSO

# ---- stage gates -----------------------------------------------------------
# Defaults run everything cheap and nothing expensive. Flip stage 3 on only
# after reading the decision-point table.
RUN_SMOKE      = True    # seconds — verify the untested runners actually run
RUN_FRONTS     = True    # minutes — exhaustive reference fronts (disk-cached)

# stage 2 — minutes each
RUN_NATIVE     = True    # NSGA-II + GDE3   <-- the decision point
RUN_BASELINES  = True    # random search + regularized evolution
RUN_CS1        = True    # Case Study 1, accuracy-only EAs
RUN_SCALARIZED = True    # scalarized EAs, both aggregations

# stage 3 — hours each
RUN_QEHVI      = False   # ~7 h
RUN_BO_CS1     = False   # ~11 h
RUN_BO_SCAL    = False   # ~35 h at 13 weights x 3 seeds x 3 datasets

# BO is the expensive one; keep it on a single aggregation. The Tchebycheff
# comparison is about the aggregation, not the optimizer, and the four EAs
# establish it already.
BO_METHOD = 'weighted_sum'
BO_ALPHAS = ALPHA_SWEEP_DENSE

RESULTS = ROOT / 'colab' / 'results' / 'MO_NAS_rev'
RESULTS.mkdir(parents=True, exist_ok=True)

print('budget      :', BUDGET, 'proposals per run')
print('alpha sweep :', ALPHA_SWEEP_DENSE, f'({len(ALPHA_SWEEP_DENSE)} weights)')
print('results     :', RESULTS)
print('stages      :', {k: v for k, v in globals().items() if k.startswith('RUN_')})

## Checkpointing

Every finished run appends one row to `manifest.csv` and is skipped on a later
pass. Delete the file (or single rows) to force a redo.

In [ ]:
MANIFEST = RESULTS / 'manifest.csv'
_done = set()
if MANIFEST.exists():
    _done = set(pd.read_csv(MANIFEST)['key'].astype(str))
    print(f'{len(_done)} runs already recorded — they will be skipped')


def _flatten(run):
    """Pull a flat row out of either runner shape (with or without 'score')."""
    s = run.get('score', {}) or {}
    cfg = run.get('config', {}) or {}
    best = run.get('best') or {}
    return {
        'n_proposals':   s.get('n_proposals',  cfg.get('n_proposals')),
        'n_unique':      s.get('n_unique',     cfg.get('n_unique')),
        'n_cache_hits':  s.get('n_cache_hits', cfg.get('n_cache_hits')),
        'hv_test':       s.get('hv_test'),
        'hv_gap_test':   s.get('hv_gap_test'),
        'igd_plus_test': s.get('igd_plus_test'),
        'front_size':    s.get('front_size'),
        'stalled':       s.get('stalled',     cfg.get('stalled')),
        'budget_met':    s.get('budget_met',  cfg.get('budget_met')),
        'sel_valid_acc': best.get('valid_acc'),
        'sel_test_acc':  best.get('test_acc'),
        'sel_flops_M':   best.get('flops_M'),
        'time_s':        run.get('total_time_s'),
        'stem':          run.get('stem'),
    }


# A fixed schema is mandatory. Appending rows whose dicts have different key
# sets writes them under the header of whichever row happened to be written
# first, silently shifting every later value into the wrong column.
COLUMNS = [
    'key', 'stamp', 'study', 'algorithm', 'dataset', 'seed', 'native',
    'scalarization', 'n_proposals', 'n_unique', 'n_cache_hits',
    'stalled', 'budget_met', 'hv_test', 'hv_gap_test', 'igd_plus_test',
    'front_size', 'sel_valid_acc', 'sel_test_acc', 'sel_flops_M', 'time_s',
    'stem',
]


def record(key, **row):
    """Append one row to the manifest immediately (crash-safe)."""
    unknown = set(row) - set(COLUMNS)
    if unknown:
        raise KeyError(f'unknown manifest column(s) {sorted(unknown)} — '
                       f'add them to COLUMNS or fix the typo')
    row = {'key': key, 'stamp': datetime.now().isoformat(timespec='seconds'),
           **row}
    df = pd.DataFrame([{c: row.get(c) for c in COLUMNS}])[COLUMNS]
    df.to_csv(MANIFEST, mode='a', header=not MANIFEST.exists(), index=False)
    _done.add(key)


def todo(key):
    return key not in _done


def manifest():
    return pd.read_csv(MANIFEST) if MANIFEST.exists() else pd.DataFrame()

## Stage 1 — reference fronts

The reference Pareto front and the accuracy upper bound stay defined on
**test** accuracy: they are the target a method is judged against, never a
signal the search may read. Enumerating all 15,625 cells takes a few minutes
per dataset, so the result is cached to disk.

In [ ]:
true_fronts, flops_ranges, hv_star, upper_bound = {}, {}, {}, {}

# The original pipeline already enumerated all 15,625 cells and saved both the
# FLOPs range and the true front. Reuse them: the reference front and the
# accuracy upper bound are defined on TEST accuracy — they are the target a
# method is judged against, never a signal the search may read — so the
# protocol change leaves them untouched and re-enumerating would only
# reproduce them. Set VERIFY_FRONTS = True to re-derive and cross-check.
LEGACY = ROOT / 'colab' / 'results' / 'MO_NAS' / 'flops'
VERIFY_FRONTS = False

if RUN_FRONTS:
    for ds in DATASETS:
        cache_file = RESULTS / f'true_front_{ds}.json'
        legacy_front = LEGACY / f'true_front_{ds}_hp{HP}.json'
        legacy_range = LEGACY / f'flops_range_{ds}_hp{HP}.json'

        if cache_file.exists():
            blob = json.loads(cache_file.read_text())
            src = 'cached'
        elif legacy_front.exists() and legacy_range.exists():
            tf = json.loads(legacy_front.read_text())
            fr = json.loads(legacy_range.read_text())
            blob = {
                'objs': [[e['f1'], e['f2']] for e in tf],
                'flops_min': fr['min_flops_M'], 'flops_max': fr['max_flops_M'],
                'best_test_acc': max(e['test_acc'] for e in tf),
                'front': tf,
            }
            cache_file.write_text(json.dumps(blob))
            src = 'reused from the original pipeline'
        else:
            t0 = time.time()
            front, objs, (fmin, fmax) = trueParetoFront(api, ds, hp=HP)
            blob = {
                'objs': objs, 'flops_min': fmin, 'flops_max': fmax,
                'best_test_acc': max(r['test_acc'] for r in front),
                'front': [{**r, 'gene': list(r['gene'])} for r in front],
            }
            cache_file.write_text(json.dumps(blob))
            src = f'enumerated in {time.time() - t0:.0f}s'

        if VERIFY_FRONTS:
            _, objs2, (fmin2, fmax2) = trueParetoFront(api, ds, hp=HP)
            same = (len(objs2) == len(blob['objs']) and
                    np.allclose(sorted(objs2), sorted(blob['objs']), atol=1e-6)
                    and abs(fmin2 - blob['flops_min']) < 1e-6
                    and abs(fmax2 - blob['flops_max']) < 1e-6)
            src += '  [verified]' if same else '  [!! MISMATCH vs enumeration]'

        # Second copy in the flat-list layout aggregate_mo_nas.py reads.
        if blob.get('front'):
            writeLegacyTrueFront(RESULTS / f'true_front_{ds}.json'.replace(
                '.json', '_legacy.json'), blob['front'],
                blob['flops_min'], blob['flops_max'])

        print(f'  {ds:16} {src}')
        true_fronts[ds]  = blob['objs']
        flops_ranges[ds] = (blob['flops_min'], blob['flops_max'])
        hv_star[ds]      = hypervolume(blob['objs'], REF_POINT)
        upper_bound[ds]  = blob['best_test_acc']

    print()
    for ds in DATASETS:
        print(f'{ds:16} front={len(true_fronts[ds]):5}  '
              f'HV*={hv_star[ds]:.4f}  best_test_acc={upper_bound[ds]:.4f}  '
              f'FLOPs=[{flops_ranges[ds][0]:.2f}, {flops_ranges[ds][1]:.2f}]')

## Stage 2 — the cheap runs

Everything here is population-based and finishes in minutes. This is the stage
that decides the paper.

In [ ]:
# --- native multi-objective: NSGA-II and GDE3 -------------------------------
if RUN_NATIVE:
    for name, fn in (('NSGA-II', runNSGA2MO), ('GDE3', runGDE3MO)):
      for ds in DATASETS:
          fmin, fmax = flops_ranges[ds]
          for seed in SEEDS_MO:
              key = f'native|{name}|{ds}|{seed}'
              if not todo(key):
                  continue
              run, _ = fn(api, dataset=ds, hp=HP, budget=BUDGET,
                          pop_size=POP_SIZE, seed=seed,
                          flops_min=fmin, flops_max=fmax,
                          true_front_objs=true_fronts[ds],
                          results_dir=str(RESULTS / ALGO_FOLDERS.get(name, name.lower()) / ds))
              record(key, study='CS2', algorithm=name, dataset=ds, seed=seed,
                     native=True, scalarization=None, **_flatten(run))
              sc = run['score']
              print(f"{name:8} {ds:16} seed={seed}  "
                    f"proposals={sc['n_proposals']:5}  "
                    f"unique={sc['n_unique']:5}  "
                    f"HV={sc['hv_test']:.4f}  "
                    f"IGD+={sc['igd_plus_test']:.5f}")

### Duplicate evaluations

Every method gets the same number of objective calls and spends them however
its own search dynamics dictate. The table below reports both counts, which is
what the reviewer asked for: how much of the budget each optimizer converted
into architectures it had not already seen.

In the published runs NSGA-II averaged ~245 distinct architectures per run and
GDE3 ~696 out of the same 1020 proposals. That difference is a property of the
variation operator, not an accounting artifact, and is worth stating explicitly
in Section 4.4 alongside the hypervolume result.

In [ ]:
df = manifest()
nat = df[df.get('native') == True] if 'native' in df else pd.DataFrame()

if len(nat):
    print('=== budget actually spent ===')
    print(nat.groupby('algorithm')[['n_proposals', 'n_unique', 'n_cache_hits']]
             .mean().round(1).to_string())

    print('\n=== quality on TEST (lower gap / IGD+ is better) ===')
    print(nat.groupby(['dataset', 'algorithm'])[
        ['hv_test', 'hv_gap_test', 'igd_plus_test']]
        .agg(['mean', 'std']).round(5).to_string())

    # Paired test across (dataset, seed) cells, as in the paper.
    if HAVE['scipy'] and {'NSGA-II', 'GDE3'} <= set(nat['algorithm']):
        from scipy.stats import wilcoxon
        piv = nat.pivot_table(index=['dataset', 'seed'],
                              columns='algorithm', values='hv_test')
        piv = piv.dropna()
        if len(piv) >= 3 and {'NSGA-II', 'GDE3'} <= set(piv.columns):
            stat, p = wilcoxon(piv['GDE3'], piv['NSGA-II'])
            wins = int((piv['GDE3'] > piv['NSGA-II']).sum())
            print(f'\n=== GDE3 vs NSGA-II on hypervolume ===')
            print(f'GDE3 higher in {wins}/{len(piv)} cells   '
                  f'Wilcoxon p = {p:.4g}')
            print('\n-> If GDE3 still wins, the confound is removed and the '
                  'claim is STRONGER than before.\n'
                  '-> If it no longer wins, the causal sentence in Section 4.4 '
                  'and part of the abstract must go.')

In [ ]:
# --- baselines: random search and regularized evolution ---------------------
if RUN_BASELINES:
    for name, fn in (('RandomSearch', runRandomSearch),
                     ('RegularizedEvolution', runRegularizedEvolution)):
        for ds in DATASETS:
            fmin, fmax = flops_ranges[ds]
            for seed in SEEDS_SO:
                key = f'baseline|{name}|{ds}|{seed}'
                if not todo(key):
                    continue
                run, _ = fn(api, dataset=ds, hp=HP, budget=BUDGET, seed=seed,
                            flops_min=fmin, flops_max=fmax,
                            true_front_objs=true_fronts[ds],
                            results_dir=str(RESULTS / ALGO_FOLDERS.get(name, name.lower()) / ds))
                record(key, study='baseline', algorithm=name, dataset=ds,
                       seed=seed, native=True, scalarization=None,
                       **_flatten(run))
            print(f'{name:22} {ds:16} done')

In [ ]:
# --- Case Study 1: accuracy-only (alpha = 1 puts zero weight on FLOPs) ------
if RUN_CS1:
    for algo in EAS:
        for ds in DATASETS:
            fmin, fmax = flops_ranges[ds]
            for seed in SEEDS_SO:
                key = f'cs1|{algo}|{ds}|{seed}'
                if not todo(key):
                    continue
                run, _ = runScalarizedMealpy(
                    api, algo, dataset=ds, hp=HP, alpha=1.0,
                    method='weighted_sum', budget=BUDGET, pop_size=POP_SIZE,
                    seed=seed,
                    flops_min=fmin, flops_max=fmax,
                    results_dir=str(RESULTS / 'cs1' / algo.lower() / ds))
                record(key, study='CS1', algorithm=algo, dataset=ds, seed=seed,
                       native=False, scalarization=None, **_flatten(run))
            print(f'CS1 {algo:8} {ds:16} done')

In [ ]:
# --- Case Study 1 scalarized: both aggregations, alpha in [0, 1] -----------
# The per-alpha winners are assembled into one front per (algorithm, seed),
# then scored on test.
def assemble(points, ds):
    fmin, fmax = flops_ranges[ds]
    span = max(fmax - fmin, 1e-12)
    objs = [[1.0 - p['test_acc'], (p['flops_M'] - fmin) / span] for p in points]
    hv = hypervolume(objs, REF_POINT)
    return {'hv_test': hv, 'hv_gap_test': hv_star[ds] - hv,
            'igd_plus_test': igdPlus(objs, true_fronts[ds]),
            'front_size': len(objs),
            'sel_test_acc': max(p['test_acc'] for p in points),
            'sel_valid_acc': max(p['valid_acc'] for p in points)}


if RUN_SCALARIZED:
    for method in SCALARIZATIONS:
        for algo in EAS:
            for ds in DATASETS:
                fmin, fmax = flops_ranges[ds]
                for seed in SEEDS_MO:
                    key = f'scal|{method}|{algo}|{ds}|{seed}'
                    if not todo(key):
                        continue
                    pts, spent = [], {'n_proposals': 0, 'n_unique': 0, 'time_s': 0.0}
                    for i, alpha in enumerate(ALPHA_SWEEP_DENSE):
                        run, _ = runScalarizedMealpy(
                            api, algo, dataset=ds, hp=HP, alpha=alpha,
                            method=method, budget=BUDGET, pop_size=POP_SIZE,
                            seed=seed * 100 + i,
                            flops_min=fmin, flops_max=fmax,
                            results_dir=str(RESULTS / method / algo.lower() / ds))
                        pts.append({**run['best'], 'alpha': alpha})
                        spent['n_proposals'] += run['config']['n_proposals']
                        spent['n_unique'] += run['config']['n_unique']
                        spent['time_s'] += run['total_time_s']
                    # The per-alpha winners are what the sweep figures plot,
                    # so keep them rather than only their aggregate.
                    sp = RESULTS / 'scalar_points'
                    sp.mkdir(parents=True, exist_ok=True)
                    (sp / f'{method}_{algo}_{ds}_seed{seed}.json').write_text(
                        json.dumps({'algorithm': algo, 'dataset': ds,
                                    'seed': seed, 'method': method,
                                    'points': pts}))
                    record(key, study='CS1-scalarized', algorithm=algo,
                           dataset=ds, seed=seed, native=False,
                           scalarization=method, **spent, **assemble(pts, ds))
                print(f'{method:14} {algo:8} {ds:16} done')

## Stage 3 — the expensive runs

Gaussian-process methods. Only start these once stage 2 has told you what the
paper says. Each cell is independently resumable.

| Cell | Runs | Rough cost |
|---|---|---|
| qEHVI | 3 datasets × 3 seeds | ~7 h |
| BO, Case Study 1 | 3 × 5 | ~11 h |
| BO, scalarized | 3 × 3 × 13 weights | ~35 h |

In [ ]:
if RUN_QEHVI:
    for ds in DATASETS:
        fmin, fmax = flops_ranges[ds]
        for seed in SEEDS_MO:
            key = f'native|qEHVI|{ds}|{seed}'
            if not todo(key):
                continue
            t0 = time.time()
            run, _ = runQEHVIMO(api, dataset=ds, hp=HP, budget=BUDGET,
                                init_points=POP_SIZE, batch_size=5,
                                mc_samples=128, num_restarts=10,
                                raw_samples=128, seed=seed,
                                flops_min=fmin, flops_max=fmax,
                                true_front_objs=true_fronts[ds],
                                results_dir=str(RESULTS / ALGO_FOLDERS['qEHVI'] / ds))
            record(key, study='CS2', algorithm='qEHVI', dataset=ds, seed=seed,
                   native=True, scalarization=None, **_flatten(run))
            print(f"qEHVI {ds:16} seed={seed}  {time.time()-t0/60:.0f} min  "
                  f"HV={run['score']['hv_test']:.4f}")

In [ ]:
if RUN_BO_CS1:
    for ds in DATASETS:
        fmin, fmax = flops_ranges[ds]
        for seed in SEEDS_SO:
            key = f'cs1|BO|{ds}|{seed}'
            if not todo(key):
                continue
            run, _ = runBOScalarized(
                api, dataset=ds, hp=HP, alpha=1.0, method='weighted_sum',
                budget=BUDGET, init_points=POP_SIZE, xi=0.01, seed=seed, flops_min=fmin, flops_max=fmax,
                results_dir=str(RESULTS / 'cs1' / 'bo' / ds))
            record(key, study='CS1', algorithm='BO', dataset=ds, seed=seed,
                   native=False, scalarization=None, **_flatten(run))
            print(f"BO CS1 {ds:16} seed={seed}  "
                  f"test={run['best']['test_acc']:.4f}  "
                  f"{run['total_time_s']/60:.0f} min")

In [ ]:
if RUN_BO_SCAL:
    for ds in DATASETS:
        fmin, fmax = flops_ranges[ds]
        for seed in SEEDS_MO:
            key = f'scal|{BO_METHOD}|BO|{ds}|{seed}'
            if not todo(key):
                continue
            pts, spent = [], {'n_proposals': 0, 'n_unique': 0, 'time_s': 0.0}
            for i, alpha in enumerate(BO_ALPHAS):
                run, _ = runBOScalarized(
                    api, dataset=ds, hp=HP, alpha=alpha, method=BO_METHOD,
                    budget=BUDGET, init_points=POP_SIZE, xi=0.01,
                    seed=seed * 100 + i,
                    flops_min=fmin, flops_max=fmax,
                    results_dir=str(RESULTS / BO_METHOD / 'bo' / ds))
                pts.append({**run['best'], 'alpha': alpha})
                spent['n_proposals'] += run['config']['n_proposals']
                spent['n_unique'] += run['config']['n_unique']
                spent['time_s'] += run['total_time_s']
                print(f'  BO {ds} seed={seed} alpha={alpha:.2f} '
                      f"({run['total_time_s']/60:.0f} min)")
            sp = RESULTS / 'scalar_points'
            sp.mkdir(parents=True, exist_ok=True)
            (sp / f'{BO_METHOD}_BO_{ds}_seed{seed}.json').write_text(
                json.dumps({'algorithm': 'BO', 'dataset': ds, 'seed': seed,
                            'method': BO_METHOD, 'points': pts}))
            record(key, study='CS1-scalarized', algorithm='BO', dataset=ds,
                   seed=seed, native=False, scalarization=BO_METHOD,
                   **spent, **assemble(pts, ds))

## Stage 4 — results

The tables that replace Tables 2, 4, 5, 7, 8 and 9 in the paper.

In [ ]:
df = manifest()
print(f'{len(df)} runs recorded\n')

print('=== Table: proposals vs unique architectures (the caching question) ===')
print(df.groupby('algorithm')[['n_proposals', 'n_unique', 'n_cache_hits']]
        .mean().round(1).to_string())

In [ ]:
print('=== Case Study 1: selection on validation, reporting on test ===')
cs1 = df[df['study'] == 'CS1']
if len(cs1):
    t = cs1.groupby(['dataset', 'algorithm'])[
        ['sel_valid_acc', 'sel_test_acc', 'n_unique', 'time_s']].agg(['mean', 'std'])
    print(t.round(4).to_string())
    print('\nupper bound (test, exhaustive):',
          {k: round(v, 4) for k, v in upper_bound.items()})
    print('\nThe gap between sel_test_acc and the upper bound is now a real '
          'generalization gap, not an artifact of having optimized on test.')

In [ ]:
print('=== Case Study 2: native vs scalarized, scored on test ===')
mo = df[df['study'].isin(['CS2', 'CS1-scalarized', 'baseline'])]
if len(mo):
    t = mo.groupby(['dataset', 'algorithm', 'scalarization'], dropna=False)[
        ['hv_test', 'hv_gap_test', 'igd_plus_test']].agg(['mean', 'std'])
    print(t.round(5).to_string())

In [ ]:
print('=== Does Tchebycheff recover the high-accuracy region? ===')
scal = df[df['study'] == 'CS1-scalarized']
if len(scal) and scal['scalarization'].nunique() > 1:
    t = scal.groupby(['dataset', 'scalarization'])[
        ['hv_test', 'igd_plus_test', 'sel_test_acc']].mean()
    print(t.round(5).to_string())
    print('\nThe old paper attributed the missing high-accuracy region to '
          'scalarization in general. With alpha=1 included and Tchebycheff '
          'alongside the weighted sum, the two causes are now separated.')

## Stage 5 — figures

Regenerates every figure the paper uses, from the results just produced.

* `utils.nas_figures` builds the Case Study 1 figures, the alpha sweeps (now
  reaching the pure-accuracy endpoint at alpha = 1) and the new
  proposals-vs-unique figures.
* `aggregate_mo_nas.py` — your existing script — builds the multi-objective
  figures and the summary table. It is unchanged apart from now accepting a
  results root, so it reads `MO_NAS_rev` instead of the old directory.

In [ ]:
FIG_DIR = RESULTS / 'figures'
FIG_DIR.mkdir(parents=True, exist_ok=True)

made = makeFigures(RESULTS, DATASETS, out_dir=FIG_DIR,
                   budget=BUDGET, pop_size=POP_SIZE,
                   methods=tuple(SCALARIZATIONS))
print(f'\n{len(made)} figures written to {FIG_DIR}')

In [ ]:
# Your existing aggregation/figure script, pointed at the new results.
import runpy

agg = ROOT / 'colab' / 'results' / 'MO_NAS' / 'aggregate_mo_nas.py'
if agg.exists():
    os.environ['MO_NAS_ROOT'] = str(RESULTS)
    os.environ['MO_NAS_FIGDIR'] = str(FIG_DIR)
    try:
        runpy.run_path(str(agg), run_name='__main__')
    except Exception as e:
        print(f'aggregate_mo_nas.py did not complete: {type(e).__name__}: {e}')
        print('(expected until the multi-objective stages have been run)')
else:
    print('aggregate_mo_nas.py not found at', agg)

In [ ]:
# Everything now in the figure directory.
for f in sorted(FIG_DIR.glob('*.pdf')):
    print(f'  {f.name:38} {f.stat().st_size / 1024:7.1f} KB')

In [ ]:
# Export a tidy CSV for the figure scripts / the paper tables.
out = RESULTS / 'revised_results.csv'
df.to_csv(out, index=False)
print('wrote', out)
print('\nremaining stages:', {k: v for k, v in globals().items()
                              if k.startswith('RUN_') and not v})